google collab depedencies

In [1]:
# !pip -q install bertopic
# !pip -q install sastrawi
# !pip -q install gensim

In [2]:
# !git clone -q -b gavriel-thesis https://github.com/ranslemus/topic_modeling_KBMI4.git
# %cd topic_modeling_KBMI4

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from hdbscan.validity import validity_index

# for linux
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

# for windows
# import umap as UMAP
# import hdbscan as HDBSCAN

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : NVIDIA GeForce GTX 1650 Ti


In [5]:
df = pd.read_csv("data/preprocessed_data_downsampled.csv")
df = df[df['year'] == 2025]
df = df[df['bank'] == "LIVIN_MANDIRI_REVIEWS"]
df.head()

,reviewId,bank,score,year,text
3,629f06db-dc19-4a6b-a526-c5fa09933ed2,LIVIN_MANDIRI_REVIEWS,2,2025,kenapa di login tidak bisa ya malah muncul tul...
27,b1409e66-396c-4151-a414-5d57f4c4e4ab,LIVIN_MANDIRI_REVIEWS,1,2025,tidak bisa digunakan tulisannya tunggu pembaha...
28,a080d0f9-09e9-457e-bb2f-b67732546821,LIVIN_MANDIRI_REVIEWS,2,2025,ini kenapa tau-tau enggak bisa diakses ya livin
29,d4d2439c-f78d-46e4-aba9-80615e2e8104,LIVIN_MANDIRI_REVIEWS,1,2025,aplksi enggak bagus gue kena tipu 25k enggak b...
32,503802d1-7efb-4095-8398-8e88828864b6,LIVIN_MANDIRI_REVIEWS,3,2025,suruh ganti sandi terus bagaimana ya


In [6]:
df["word_count"] = df["text"].astype(str).str.split().apply(len)
df = df[df["word_count"] >= 5].reset_index(drop=True)
print(f"Total documents setelah filter: {len(df):,}")

Total documents setelah filter: 11,112


In [7]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 11,112


# indoSBERT-large

In [8]:
from sentence_transformers import SentenceTransformer

# Gunakan SimCSE untuk menekan anisotropy dan merapatkan klaster
embedding_model = SentenceTransformer("denaya/indoSBERT-large", device=device)

embeddings = embedding_model.encode(
    documents,
    batch_size=64, 
    show_progress_bar=True
)

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.23k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Dense/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.05MB            

2_Dense/pytorch_model.bin: downloading bytes:           |  0.00B            

Batches:   0%|          | 0/174 [00:00<?, ?it/s]

# BERTopic

In [9]:
# embeddings = np.load("indobert_embeddings.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (11112, 256)


stop words

In [10]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
from nltk.corpus import stopwords as nltk_stopwords
from bertopic.vectorizers import ClassTfidfTransformer

sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

# Pure stopwords gabungan (NLTK + Sastrawi)
pure_stopwords = list(set(nltk_stopwords.words('indonesian')).union(set(sastrawi_stopwords)))


vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    stop_words=pure_stopwords,
    token_pattern=r"(?u)\b[^\d\W]+\b",
    min_df=5  # Untuk 60k data, min_df=5 efektif membuang kata typo langka
)

# Strict c-TF-IDF Transformer untuk memotong frequent words antar-klaster
ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True
)

baseline UMAP for testing purpose

In [12]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=10,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [13]:
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

In [14]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    ctfidf_model=ctfidf_model,
    verbose=True
)

In [15]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-12 11:42:09,605 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-12 11:42:18,141 - BERTopic - Dimensionality - Completed ✓
2026-08-12 11:42:18,151 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-12 11:42:19,244 - BERTopic - Cluster - Completed ✓
2026-08-12 11:42:19,302 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-12 11:42:20,720 - BERTopic - Representation - Completed ✓


outliers removal

# Evaluation for Topic Quality

Basic Statistics

In [16]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,5968,-1_cabang_livin mandiri_nomor_apk,"[cabang, livin mandiri, nomor, apk, ribet, apl...",[parah sangat merugikan nasabah pengguna aplik...
1,0,504,0_malam_pemeliharaan_maintenance_kalo malam,"[malam, pemeliharaan, maintenance, kalo malam,...",[selalu di jam 11 sudah enggak bisa tf masa to...
2,1,460,1_biaya_potongan_admin_biaya admin,"[biaya, potongan, admin, biaya admin, ribu, ke...",[kenapa ya tiap bayar virtual account sekarang...
3,2,386,2_wajah_verifikasi wajah_verifikasi_wajah gagal,"[wajah, verifikasi wajah, verifikasi, wajah ga...",[susah amat verifikasi wajah sudah kedip mata ...
4,3,375,3_pinjaman_saldo_berkurang_terpotong,"[pinjaman, saldo, berkurang, terpotong, uang, ...",[tidak bisa transfer uang ke bank lain dan top...
5,4,344,4_update_lemot_update update_penuh,"[update, lemot, update update, penuh, update m...",[apk lemot banget loading nya bagusan yang lam...
6,5,331,5_menunggu_tunggu_minggu_hasil,"[menunggu, tunggu, minggu, hasil, pengiriman, ...",[pemeliharaan rutin dari jam 12 malam sampai j...
7,6,286,6_login login_login susah_login aplikasi_susah...,"[login login, login susah, login aplikasi, sus...",[susah login aplikasi sedang dalam perbaikan t...
8,7,246,7_sandi_password_lupa_pin,"[sandi, password, lupa, pin, pasword, ganti sa...",[saya lupa sandi sehingga akun saya keluar sen...
9,8,205,8_android_sistem operasi_operasi_mendukung,"[android, sistem operasi, operasi, mendukung, ...",[aplikasi livin by mandiri tidak dapat dipakai...


In [17]:
num_topics = len(topic_info) - 1

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 33
Outliers            : 5,968
Outlier Percentage  : 53.71%


Topic Size

In [18]:
topic_info[["Topic","Count"]]

,Topic,Count
0,-1,5968
1,0,504
2,1,460
3,2,386
4,3,375
5,4,344
6,5,331
7,6,286
8,7,246
9,8,205


Top Words

In [19]:
top_10_topics = topic_model.get_topic_info()
top_10_topics = top_10_topics[top_10_topics.Topic != -1].nlargest(10, "Count")

for _, row in top_10_topics.iterrows():
    topic_id = row['Topic']
    doc_count = row['Count']

    print("=" * 80)
    print(f"TOPIC {topic_id} | JUMLAH DOKUMEN: {doc_count}")
    print("=" * 80)

    # Menampilkan word-score pair bawaan BERTopic (c-TF-IDF scores)
    words_with_scores = topic_model.get_topic(topic_id)
    for word, score in words_with_scores:
        print(f"  - {word:<20} : {score:.4f}")
    print()

TOPIC 0 | JUMLAH DOKUMEN: 504
  - malam                : 0.5507
  - pemeliharaan         : 0.4886
  - maintenance          : 0.4406
  - kalo malam           : 0.4257
  - jam                  : 0.3936
  - jam malam            : 0.3667
  - rutin                : 0.3379
  - urgent               : 0.3353
  - malam gangguan       : 0.3120
  - gangguan             : 0.3114

TOPIC 1 | JUMLAH DOKUMEN: 460
  - biaya                : 0.4597
  - potongan             : 0.4243
  - admin                : 0.3752
  - biaya admin          : 0.3551
  - ribu                 : 0.3417
  - kena                 : 0.3230
  - administrasi         : 0.3104
  - saldo                : 0.2960
  - kena biaya           : 0.2851
  - potong               : 0.2843

TOPIC 2 | JUMLAH DOKUMEN: 386
  - wajah                : 0.6138
  - verifikasi wajah     : 0.6124
  - verifikasi           : 0.5375
  - wajah gagal          : 0.4895
  - muka                 : 0.4190
  - gagal                : 0.3416
  - mengikuti           

Representative Reviews

In [20]:
# Ambil info topik dan urutkan berdasarkan jumlah dokumen terbesar (kecuali outlier -1)
topic_info = topic_model.get_topic_info()
top_10_topics = topic_info[topic_info.Topic != -1].nlargest(10, "Count")["Topic"].tolist()

print("=== TOP 10 TOPIK PALING REPRESENTATIF ===")

for topic_id in top_10_topics:
    # Ambil ukuran klaster asli
    cluster_size = topic_info.loc[topic_info.Topic == topic_id, "Count"].values[0]

    # Ambil kata kunci utama topik untuk mempermudah pembacaan aspek
    keywords = ", ".join([w for w, _ in topic_model.get_topic(topic_id)[:5]])

    # Ambil dokumen yang secara matematis paling dekat dengan centroid klaster (Bawaan BERTopic)
    rep_docs = topic_model.get_representative_docs(topic_id)

    print("\n" + "=" * 120)
    print(f"TOPIC {topic_id} | CLUSTER SIZE: {cluster_size}")
    print(f"KEYWORDS : {keywords}")
    print("=" * 120)

    # BERTopic menyimpan maksimum 3 representative docs per topik secara default
    for i, doc in enumerate(rep_docs, 1):
        print(f"{i}. {doc}")

=== TOP 10 TOPIK PALING REPRESENTATIF ===

TOPIC 0 | CLUSTER SIZE: 504
KEYWORDS : malam, pemeliharaan, maintenance, kalo malam, jam
1. selalu di jam 11 sudah enggak bisa tf masa tolong perbaiki ya jadi susah banget kalo harus menunggu beberapa jam buat pemeliharaan terus kayak begitu padahal kebutuhan orang beda waktu jadinya untuk menyimpan saldo di livin lebih dipikir lagi karena begitu di jam malam sudah enggak bisa tf belum lagi suka eror
2. untuk saya yang mulai beraktifitas malam hari karena memang jadwal pekerjaan di malam hari livin ini memang enggak bisa di andelin setiap tengah malam sampai jam 3 mbanking enggak bisa di apa2in mau top up emoney enggak bisa mau tf enggak bisa pokoknya enggak bisa diapa-apai kalo sudah tengah malam bahkan kadang jam 11 pun sudah maintenance entah sampai kapan selesai nya maintenance setiap hari
3. saya di jam 01 42 malam laper menunggu sampai kapan pemeliharaan rutin enggak harus malam kan jam jam orang laper mau belanja mau ini mau itu harus m

silhoutte score

In [21]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : 0.5075


In [22]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info["Topic"]:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]

    topic_words.append(words)

flat_words = list(chain.from_iterable(topic_words))

unique_words = len(set(flat_words))
total_words = len(flat_words)

topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.8485


NPMI

In [23]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [24]:
doc.split()

['kenapa',
 'setiap',
 'mau',
 'transaksi',
 'muncul',
 'pengaturan',
 'jam',
 'dan',
 'tanggal',
 'otomatis',
 'sesuai',
 'waktu',
 'sudah',
 'beberapa',
 'waktu',
 'ini',
 'tetap',
 'saja',
 'seperti',
 'itu']

In [25]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [26]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)

top_n = 10
topic_words = []

for topic in topic_info["Topic"]:

    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
        if word in dictionary.token2id
    ]

    if len(words) >= 2:
        topic_words.append(words)

print(f"Valid Topics for NPMI: {len(topic_words)}")

Valid Topics for NPMI: 33


In [27]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

NPMI : -0.0884


In [28]:
per_topic_npmi = coherence_model.get_coherence_per_topic()

npmi_df = pd.DataFrame({
    "Topic": [
        t for t in topic_info["Topic"]
        if t != -1
    ],
    "NPMI": per_topic_npmi
})

npmi_df.sort_values("NPMI", ascending=False).head(10)

,Topic,NPMI
9,9,0.251980
8,8,0.239740
1,1,0.175339
27,27,0.136602
0,0,0.127452
11,11,0.108729
19,19,0.104949
2,2,0.093401
14,14,0.072112
7,7,0.026656


In [29]:
worst_topics = (
    npmi_df
    .sort_values("NPMI", ascending=True)
    .head(10)["Topic"]
    .tolist()
)

for topic_id in worst_topics:
    print("\n" + "=" * 80)
    print(f"TOPIC {topic_id}")
    print("=" * 80)

    print("NPMI:",
          npmi_df.loc[
              npmi_df["Topic"] == topic_id,
              "NPMI"
          ].iloc[0]
    )

    print("Words:")
    print(topic_model.get_topic(topic_id))

    print("\nRepresentative documents:")
    print(
        topic_model.get_representative_docs(topic_id)[:5]
    )


TOPIC 15
NPMI: -0.4056762363222326
Words:
[('livin livin', np.float64(0.5298026879097899)), ('livin update', np.float64(0.5028651478760703)), ('nih livin', np.float64(0.45905097479149126)), ('livin susah', np.float64(0.4105834342192304)), ('update livin', np.float64(0.402909488410901)), ('lelet', np.float64(0.3894668686371005)), ('pakai nya', np.float64(0.3876055985843616)), ('kendala livin', np.float64(0.3792048286869231)), ('livin gangguan', np.float64(0.37093614751372567)), ('buka livin', np.float64(0.3649557199255326))]

Representative documents:
['mau buka livin saja susah banget muter2 doang loadingnya enggak masuk2', 'tolong kasih tau min sudah beberapa hari ini livin enggak bisa di akses setelah jam 18 00 afa kendala apa ya sebelum nya enggak pernah kayak begini', 'apa livin susah di buka tampilan layar selalu warna hijau tanpa bisa di akses padahal sudah sering di upgrade tak seperti bank bank sebelah begitu gampang di gunakan tolong mi']

TOPIC 26
NPMI: -0.3954088721326195
W

In [30]:
per_topic = np.array(
    coherence_model.get_coherence_per_topic()
)

overall = coherence_model.get_coherence()

print("Gensim overall :", overall)
print("Mean per-topic :", per_topic.mean())
print("Difference     :", overall - per_topic.mean())

Gensim overall : -0.08838570632703459
Mean per-topic : -0.08838570632703459
Difference     : 0.0


# Evaluation for Clustering Quality


DBCV

In [31]:
mask = np.array(topics) != -1
X = topic_model.umap_model.embedding_[mask].astype(np.float64)
labels = np.array(topics)[mask]

dbcv_score = validity_index(X, labels)
print(f"DBCV : {dbcv_score:.4f}")

DBCV : 0.1523


In [32]:
import pandas as pd
from scipy.stats import chi2_contingency

df["topic"] = topics

# 1. Baseline: proporsi tiap bank di keseluruhan korpus
baseline = df["bank"].value_counts(normalize=True) * 100
print("Proporsi bank di keseluruhan korpus (baseline):")
print(baseline.round(2))
print()

# 2. Proporsi tiap bank DI DALAM tiap topik
crosstab = pd.crosstab(df["topic"], df["bank"], normalize="index") * 100
crosstab = crosstab.round(2)

# 3. Hitung "lift" = proporsi di topik / proporsi baseline
#    >1 artinya over-represented di topik itu, <1 artinya under-represented
lift = crosstab.copy()
for bank in baseline.index:
    lift[bank] = crosstab[bank] / baseline[bank]

# 4. Tandai topik yang "njomplang" (deviasi lift > 1.5x atau < 0.5x dari baseline)
def flag_imbalance(row):
    return any(row > 1.5) or any(row < 0.5)

lift["is_imbalanced"] = lift[baseline.index].apply(flag_imbalance, axis=1)

# gabung count per topik biar gampang liat mana yang topik "besar" (bukan cuma noise kecil)
topic_sizes = df[df["topic"] != -1]["topic"].value_counts()
lift["topic_size"] = lift.index.map(topic_sizes)

result = lift[lift.index != -1].sort_values("is_imbalanced", ascending=False)
print(result[list(baseline.index) + ["is_imbalanced", "topic_size"]])

Proporsi bank di keseluruhan korpus (baseline):
bank
LIVIN_MANDIRI_REVIEWS    100.0
Name: proportion, dtype: float64

bank   LIVIN_MANDIRI_REVIEWS  is_imbalanced  topic_size
topic                                                  
0                        1.0          False       504.0
17                       1.0          False        85.0
31                       1.0          False        54.0
30                       1.0          False        54.0
29                       1.0          False        55.0
28                       1.0          False        56.0
27                       1.0          False        56.0
26                       1.0          False        62.0
25                       1.0          False        62.0
24                       1.0          False        64.0
23                       1.0          False        66.0
22                       1.0          False        74.0
21                       1.0          False        77.0
20                       1.0          Fals

checking outliers

In [33]:
# import itertools

# param_grid = {
#     "min_cluster_size": [30, 50, 75],
#     "min_samples": [10, 15, 20],
#     "cluster_selection_method": ["eom", "leaf"],
# }

# results = []
# combos = list(itertools.product(*param_grid.values()))
# print(f"Total kombinasi: {len(combos)}")

# for mcs, ms, method in combos:
#     hdbscan_test = HDBSCAN(
#         min_cluster_size=mcs, min_samples=ms, metric="euclidean",
#         cluster_selection_method=method, prediction_data=True,
#     )
#     tm = BERTopic(
#         embedding_model=None, calculate_probabilities=False,
#         vectorizer_model=vectorizer_model, umap_model=umap_model,
#         hdbscan_model=hdbscan_test, verbose=False,
#     )
#     tpcs, _ = tm.fit_transform(documents, embeddings)

#     ti = tm.get_topic_info()
#     n_topics = len(ti) - 1
#     outlier_pct = (np.array(tpcs) == -1).sum() / len(tpcs) * 100
#     max_share = ti[ti.Topic != -1]["Count"].max() / len(tpcs) * 100 if n_topics > 0 else 0
#     mask = np.array(tpcs) != -1
#     sil = silhouette_score(tm.umap_model.embedding_[mask], np.array(tpcs)[mask]) if len(set(np.array(tpcs)[mask])) > 1 else float("nan")

#     row = {"min_cluster_size": mcs, "min_samples": ms, "method": method,
#            "topics": n_topics, "outlier_%": round(outlier_pct, 2),
#            "max_topic_share_%": round(max_share, 2), "silhouette": round(sil, 4)}
#     results.append(row)
#     print(row)

# results_df = pd.DataFrame(results).sort_values("outlier_%")
# results_df

In [34]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

topics_array = np.array(topics)

outlier_idx = np.where(topics_array == -1)[0]
clustered_idx = np.where(topics_array != -1)[0]

# Ambil sample outlier
rng = np.random.default_rng(42)

sample_outlier_idx = rng.choice(
    outlier_idx,
    size=min(1000, len(outlier_idx)),
    replace=False
)

outlier_embeddings = embeddings[sample_outlier_idx]
clustered_embeddings = embeddings[clustered_idx]

# Similarity outlier -> ALL clustered documents
similarity_matrix = cosine_similarity(
    outlier_embeddings,
    clustered_embeddings
)

nearest_cluster_similarity = similarity_matrix.max(axis=1)

print(pd.Series(nearest_cluster_similarity).describe())

count    1000.000000
mean        0.779772
std         0.059997
min         0.441070
25%         0.748516
50%         0.792255
75%         0.820909
max         0.916517
dtype: float64
